# Random Number Generator on a Local Simulator

&copy; 2026 by [Damir Cavar](http://damir.cavar.me/)

This is an example of a random number generator using two quantum bits and a Bell circuit, updated for Qiskit 2.x.

Prerequisites can be installed using the following code:

In [ ]:
!pip install -U "qiskit>=2.0"
!pip install -U "qiskit-ibm-runtime>=0.30"
!pip install -U "qiskit-aer>=0.15"

# Local Simulated Random Generator

In Qiskit 2.x we run circuits on a backend through the V2 `Sampler` primitive; `backend.run()` is no longer the supported path for real-device execution. The same code works against `FakeManilaV2` (a noise-model-backed fake backend) by passing the fake backend to `SamplerV2` via `mode=`.

We need the following imports:

In [ ]:
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit_ibm_runtime.fake_provider import FakeManilaV2

We define a 2-qubit Bell circuit with measurements into a classical register:

In [ ]:
q = QuantumRegister(2, 'q')
c = ClassicalRegister(2, 'c')
circuit = QuantumCircuit(q, c)
circuit.h(q[0])
circuit.cx(q[0], q[1])
circuit.measure(q, c)

Use the `FakeManilaV2` backend and produce an ISA circuit with the preset pass manager:

In [ ]:
fake_manila = FakeManilaV2()
pm = generate_preset_pass_manager(backend=fake_manila, optimization_level=1)
isa_qc = pm.run(circuit)

We can print and visualize the circuit:

In [ ]:
print(isa_qc)

global phase: π/4
         ┌─────────┐┌────┐┌─────────┐     ┌─┐   
q_0 -> 0 ┤ Rz(π/2) ├┤ √X ├┤ Rz(π/2) ├──■──┤M├───
         └─────────┘└────┘└─────────┘┌─┴─┐└╥┘┌─┐
q_1 -> 1 ────────────────────────────┤ X ├─╫─┤M├
                                     └───┘ ║ └╥┘
    c: 2/══════════════════════════════════╩══╩═
                                           0  1 


Create a `SamplerV2` bound to the fake backend and submit the ISA circuit with 100 shots:

In [ ]:
sampler = Sampler(mode=fake_manila)
job = sampler.run([isa_qc], shots=100)

Print the result. In Qiskit 2.x the V2 `Sampler` returns per-PUB results and the measurement bitstring counts live under the classical register's name (here, `c`):

In [ ]:
result = job.result()
pub_result = result[0]
counts = pub_result.data.c.get_counts()
print('Result:', counts)

Result: {'00': 51, '11': 45, '01': 2, '10': 2}


(C) 2026 by [Damir Cavar](http://damir.cavar.me/)